### Chatbot And RAG Evaluation (with Langfuse)

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This notebook shows how to evaluate your chatbot / RAG applications using **Langfuse** (the open-source alternative to LangSmith). You'll learn:

1. How to create test datasets in Langfuse
2. How to run your application on those datasets with `run_experiment`
3. How to measure performance with LLM-as-a-judge evaluators (correctness, relevance, groundedness, retrieval relevance)

#### Overview
A typical evaluation workflow has three steps:

1. Create a dataset with questions and expected answers (`create_dataset` + `create_dataset_item`)
2. Run your application on each item via a `task` function
3. Attach `evaluators` that score each output; results are traced and aggregated in the Langfuse UI

For this tutorial, we'll build and evaluate a bot that answers questions about a few of Lilian Weng's blog posts.

> **Langfuse vs LangSmith cheat-sheet**
> | LangSmith | Langfuse |
> |---|---|
> | `Client()` | `get_client()` |
> | `client.create_dataset` / `create_examples` | `create_dataset` / `create_dataset_item` |
> | `@traceable` | `@observe` |
> | `wrappers.wrap_openai` | `from langfuse.openai import openai` |
> | `client.evaluate(target, data, evaluators)` | `client.run_experiment(name, data, task, evaluators)` |
> | evaluator returns `bool` | evaluator returns `Evaluation(name=, value=, comment=)` |

### Chatbot Evaluation

In [8]:
import os
from dotenv import load_dotenv
from langfuse import get_client, observe, Evaluation

load_dotenv()

# Langfuse credentials (put these in your .env file)
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"] = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = get_client()
assert client.auth_check(), "Langfuse auth failed - check LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_HOST"
print("Langfuse client authenticated")

Langfuse client authenticated


In [9]:
# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"

examples = [
    {"inputs": {"question": "What is LangChain?"},
     "outputs": {"answer": "A framework for building LLM applications"}},
    {"inputs": {"question": "What is LangSmith?"},
     "outputs": {"answer": "A platform for observing and evaluating LLM applications"}},
    {"inputs": {"question": "What is OpenAI?"},
     "outputs": {"answer": "A company that creates Large Language Models"}},
    {"inputs": {"question": "What is Google?"},
     "outputs": {"answer": "A technology company known for search"}},
    {"inputs": {"question": "What is Mistral?"},
     "outputs": {"answer": "A company that creates Large Language Models"}},
]

# create_dataset upserts by name -> safe to re-run
client.create_dataset(name=dataset_name)

# Passing a stable `id` (the question) makes re-runs upsert instead of
# creating duplicate items.
for ex in examples:
    client.create_dataset_item(
        dataset_name=dataset_name,
        id=ex["inputs"]["question"],
        input=ex["inputs"],
        expected_output=ex["outputs"],
    )

print(f"Created dataset '{dataset_name}' with {len(examples)} items")

Created dataset 'Chatbots Evaluation' with 5 items


### Define Metrics (LLM As A Judge)

In [10]:
# Langfuse drop-in wrapper for the OpenAI SDK: every call below is auto-traced.
from langfuse.openai import openai

openai_client = openai.OpenAI()

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

# A Langfuse evaluator receives the item's `input`, the task `output`,
# the `expected_output`, and `metadata` as keyword arguments, and returns
# one (or a list of) Evaluation(name=..., value=..., comment=...) object(s).
def correctness(*, input, output, expected_output, metadata=None, **kwargs):
    """LLM-as-judge: is the predicted answer correct vs the reference answer?"""
    user_content = f"""You are grading the following question:
{input['question']}
Here is the real answer:
{expected_output['answer']}
You are grading the following predicted answer:
{output}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content

    return Evaluation(name="correctness", value=response.strip() == "CORRECT", comment=response)

In [11]:
# Concision - checks whether the output is less than 2x the length of the expected answer.
def concision(*, input, output, expected_output, metadata=None, **kwargs):
    return Evaluation(
        name="concision",
        value=int(len(output) < 2 * len(expected_output["answer"])),
    )

### Run Evaluations

In [12]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "gpt-4o-mini", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [13]:
# The task function is called once per dataset item.
# `item.input` is the dict we stored -> {"question": ...}
def task(*, item, **kwargs) -> str:
    return my_app(item.input["question"])

In [14]:
dataset = client.get_dataset(dataset_name)

result = client.run_experiment(
    name="openai-4o-mini-chatbot",
    data=dataset.items,
    task=task,
    evaluators=[correctness, concision],
)

print(result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: openai-4o-mini-chatbot
📋 Run name: openai-4o-mini-chatbot - 2026-07-02T02:39:21.706541Z
5 items
Evaluations:
  • concision
  • correctness

Average Scores:
  • concision: 0.200
  • correctness: 0.600

🔗 Dataset Run:
   http://localhost:3000/project/cmpkfwk680006qg07vc8nioem/datasets/cmr2w6dw40001l1070x5mm1yf/runs/c5e97703-5628-467d-9c31-06ad5ed67bf2


In [ ]:
# Same dataset, different model -> comparable experiment run
def task_turbo(*, item, **kwargs) -> str:
    return my_app(item.input["question"], model="gpt-4-turbo")

result = client.run_experiment(
    name="openai-4-turbo-chatbot",
    data=dataset.items,
    task=task_turbo,
    evaluators=[correctness, concision],
)

print(result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: openai-4-turbo-chatbot
📋 Run name: openai-4-turbo-chatbot - 2026-07-02T02:39:31.330665Z
5 items
Evaluations:
  • concision
  • correctness

Average Scores:
  • concision: 0.200
  • correctness: 0.600

🔗 Dataset Run:
   http://localhost:3000/project/cmpkfwk680006qg07vc8nioem/datasets/cmr2w6dw40001l1070x5mm1yf/runs/5f3f5c29-8d7a-4e86-98bf-088ea398f144


### Evaluation For RAG

In [16]:
## RAG retriever
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to an in-memory vector store using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# Turn the vector store into a retrieval component
retriever = vectorstore.as_retriever(k=6)

C:\Users\Tho Le\AppData\Local\Temp\ipykernel_23840\2753724709.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\ai_learning\krisknaik\ultimate-rag-bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [17]:
retriever.invoke("what is agents")

[Document(id='7e922347-6c50-402b-b9cc-548239ae6d41', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [18]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000023A4BFD8050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023A4BFB17F0>, root_client=<openai.OpenAI object 

In [ ]:
# @observe traces this function (and the nested retriever + llm calls) in Langfuse,
# the Langfuse equivalent of LangSmith's @traceable.
@observe()
def rag_bot(question: str) -> dict:
    ## Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions. 
    Use the following source documents to answer the user's questions. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ## llm invoke
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents": docs}

In [20]:
rag_bot("What is agents")

{'answer': 'Agents, in this context, refer to autonomous entities powered by large language models (LLMs) that simulate human-like behavior and interactions. They are designed to perform tasks, make decisions, and engage in planning and reflection based on past experiences in a simulated environment. Examples include generative agents that create believable simulations for interactive applications.',
 'documents': [Document(id='7e922347-6c50-402b-b9cc-548239ae6d41', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-p

### Dataset

In [21]:
# Define the examples for the RAG dataset
rag_dataset_name = "RAG Test Evaluation"

rag_examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    },
]

client.create_dataset(name=rag_dataset_name)
for ex in rag_examples:
    client.create_dataset_item(
        dataset_name=rag_dataset_name,
        id=ex["inputs"]["question"],
        input=ex["inputs"],
        expected_output=ex["outputs"],
    )

print(f"Created dataset '{rag_dataset_name}' with {len(rag_examples)} items")

Created dataset 'RAG Test Evaluation' with 3 items


### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [22]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

## Correctness output schema
class CorrectnessGrade(TypedDict):
    # Putting the explanation before the boolean forces the model to reason first.
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

grader_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

## evaluator - note: for the RAG task, `output` is the dict {"answer", "documents"}
def correctness(*, input, output, expected_output, metadata=None, **kwargs):
    """An evaluator for RAG answer accuracy"""
    answers = f"""QUESTION: {input['question']}
GROUND TRUTH ANSWER: {expected_output['answer']}
STUDENT ANSWER: {output['answer']}"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return Evaluation(name="correctness", value=grade["correct"], comment=grade["explanation"])

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference answer. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [23]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

def relevance(*, input, output, expected_output=None, metadata=None, **kwargs):
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {input['question']}\nSTUDENT ANSWER: {output['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return Evaluation(name="relevance", value=grade["relevant"], comment=grade["explanation"])

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [24]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

def groundedness(*, input, output, expected_output=None, metadata=None, **kwargs):
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in output["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {output['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return Evaluation(name="groundedness", value=grade["grounded"], comment=grade["explanation"])

### Retrieval Relevance: Retrieved docs vs input

In [25]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    RetrievalRelevanceGrade, method="json_schema", strict=True
)

def retrieval_relevance(*, input, output, expected_output=None, metadata=None, **kwargs):
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in output["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {input['question']}"
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return Evaluation(name="retrieval_relevance", value=grade["relevant"], comment=grade["explanation"])

### Run the evaluation

In [26]:
# The task wraps our traced rag_bot; its dict return is passed to every evaluator as `output`.
def rag_task(*, item, **kwargs) -> dict:
    return rag_bot(item.input["question"])

rag_dataset = client.get_dataset(rag_dataset_name)

result = client.run_experiment(
    name="rag-doc-relevance",
    data=rag_dataset.items,
    task=rag_task,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    metadata={"version": "LCEL context, gpt-4o-mini"},
)

print(result.format())

# Make sure everything is sent before the notebook exits.
client.flush()

Individual Results: Hidden (3 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: rag-doc-relevance
📋 Run name: rag-doc-relevance - 2026-07-02T02:41:01.058108Z
3 items
Evaluations:
  • groundedness
  • relevance
  • retrieval_relevance
  • correctness

Average Scores:
  • groundedness: 0.667
  • relevance: 1.000
  • retrieval_relevance: 0.667
  • correctness: 1.000

🔗 Dataset Run:
   http://localhost:3000/project/cmpkfwk680006qg07vc8nioem/datasets/cmr2wdvkx000pl1073u7im26x/runs/1359dd62-1681-4569-9d95-6219d7b0ff54
